# Exploratory Data Analysis — Flight Data 2024

**Goal:** Understand the raw data before writing any pipeline code.

**Sections:**
1. Load Data
2. Basic Info
3. Missing Value Analysis
4. Target Variable Analysis
5. Distribution Plots
6. Delay by Category (carrier, time, day, route)
7. Correlation Analysis
8. Key Findings & Feature Ideas

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('../data/raw/flight_data_2024.csv', low_memory=False)
print(f'Rows:    {len(df):,}')
print(f'Columns: {len(df.columns)}')

## 2. Basic Info

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} unique values')

## 3. Missing Value Analysis

In [ ]:
null_pct = (df.isnull().mean() * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)

print('Columns with missing values:\n')
for col, pct in null_pct.items():
    bar = '█' * int(pct / 2)
    print(f'{col:<30} {pct:>6.2f}%  {bar}')

In [ ]:
plt.figure(figsize=(10, 6))
null_pct.plot(kind='barh', color='steelblue')
plt.xlabel('Null %')
plt.title('Missing Value % per Column')
plt.tight_layout()
plt.show()

## 4. Target Variable Analysis

Our prediction target is `arr_delay` — arrival delay in minutes.

We will create a binary target `is_delayed` using the FAA standard: a flight is delayed if it arrives **more than 15 minutes late**.

In [ ]:
df_active = df[df['cancelled'] == 0].copy()
print(f'Total flights:     {len(df):,}')
print(f'Cancelled flights: {df["cancelled"].sum():,} ({df["cancelled"].mean():.1%})')
print(f'Active flights:    {len(df_active):,}')

In [ ]:
df_active['is_delayed'] = (df_active['arr_delay'] > 15).astype(int)

delay_rate = df_active['is_delayed'].mean()
print(f'Delayed flights: {df_active["is_delayed"].sum():,} ({delay_rate:.1%})')
print(f'On-time flights: {(df_active["is_delayed"] == 0).sum():,} ({1 - delay_rate:.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(
    df_active['is_delayed'].value_counts(),
    labels=['On Time', 'Delayed'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c']
)
axes[0].set_title('Flight Delay Distribution')

clipped = df_active['arr_delay'].clip(-60, 300)
axes[1].hist(clipped, bins=100, color='steelblue', edgecolor='none')
axes[1].axvline(15, color='red', linestyle='--', label='15 min threshold')
axes[1].axvline(0, color='orange', linestyle='--', label='On time')
axes[1].set_xlabel('Arrival Delay (minutes)')
axes[1].set_title('Distribution of Arrival Delay')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Arrival delay statistics (minutes):')
print(df_active['arr_delay'].describe().round(1))
print(f'\nFlights that arrived EARLY: {(df_active["arr_delay"] < 0).mean():.1%}')

## 5. Distribution Plots — Key Numerical Columns

In [ ]:
num_cols = ['dep_delay', 'arr_delay', 'distance', 'air_time', 'taxi_out', 'taxi_in']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df_active[col].dropna().clip(
        df_active[col].quantile(0.01),
        df_active[col].quantile(0.99)
    )
    axes[i].hist(data, bins=80, color='steelblue', edgecolor='none')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Numerical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Delay by Category

Here we ask: **which factors are associated with higher delay rates?**

For hour, day, and month charts — the bar color is value-driven: **deep red = highest delay rate, deep blue = lowest**.

In [ ]:
# Delay rate by airline
carrier_delay = (
    df_active.groupby('op_unique_carrier')['is_delayed']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
carrier_delay.columns = ['carrier', 'delay_rate']

plt.figure(figsize=(14, 5))
sns.barplot(data=carrier_delay, x='carrier', y='delay_rate', palette='Reds_r')
plt.axhline(delay_rate, color='blue', linestyle='--', label=f'Overall avg ({delay_rate:.1%})')
plt.ylabel('Delay Rate')
plt.title('Delay Rate by Airline')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by hour of day
# crs_dep_time is stored as integer like 1430 meaning 14:30 — divide by 100 to get the hour
df_active['dep_hour'] = df_active['crs_dep_time'] // 100

hour_delay = (
    df_active.groupby('dep_hour')['is_delayed']
    .mean()
    .reset_index()
)

# Color each bar by its value — highest delay rate = deep red, lowest = deep blue
norm = mcolors.Normalize(vmin=hour_delay['is_delayed'].min(), vmax=hour_delay['is_delayed'].max())
colors = [cm.RdBu_r(norm(v)) for v in hour_delay['is_delayed']]

plt.figure(figsize=(14, 5))
plt.bar(hour_delay['dep_hour'], hour_delay['is_delayed'], color=colors)
plt.axhline(delay_rate, color='black', linestyle='--', label=f'Overall avg ({delay_rate:.1%})')
plt.xlabel('Departure Hour')
plt.ylabel('Delay Rate')
plt.title('Delay Rate by Hour of Day')
plt.xticks(hour_delay['dep_hour'])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by day of week — 1=Monday, 7=Sunday
day_names = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
day_delay = (
    df_active.groupby('day_of_week')['is_delayed']
    .mean()
    .reset_index()
)
day_delay['day_name'] = day_delay['day_of_week'].map(day_names)

norm = mcolors.Normalize(vmin=day_delay['is_delayed'].min(), vmax=day_delay['is_delayed'].max())
colors = [cm.RdBu_r(norm(v)) for v in day_delay['is_delayed']]

plt.figure(figsize=(10, 5))
plt.bar(day_delay['day_name'], day_delay['is_delayed'], color=colors)
plt.axhline(delay_rate, color='black', linestyle='--', label=f'Overall avg ({delay_rate:.1%})')
plt.xlabel('Day of Week')
plt.ylabel('Delay Rate')
plt.title('Delay Rate by Day of Week')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by month
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
month_delay = (
    df_active.groupby('month')['is_delayed']
    .mean()
    .reset_index()
)
month_delay['month_name'] = month_delay['month'].map(month_names)

norm = mcolors.Normalize(vmin=month_delay['is_delayed'].min(), vmax=month_delay['is_delayed'].max())
colors = [cm.RdBu_r(norm(v)) for v in month_delay['is_delayed']]

plt.figure(figsize=(12, 5))
plt.bar(month_delay['month_name'], month_delay['is_delayed'], color=colors)
plt.axhline(delay_rate, color='black', linestyle='--', label=f'Overall avg ({delay_rate:.1%})')
plt.xlabel('Month')
plt.ylabel('Delay Rate')
plt.title('Delay Rate by Month')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 busiest origin airports and their delay rates
top_airports = df_active['origin'].value_counts().head(15).index
airport_delay = (
    df_active[df_active['origin'].isin(top_airports)]
    .groupby('origin')['is_delayed']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(14, 5))
sns.barplot(data=airport_delay, x='origin', y='is_delayed', palette='Reds_r')
plt.axhline(delay_rate, color='blue', linestyle='--', label=f'Overall avg ({delay_rate:.1%})')
plt.xlabel('Origin Airport')
plt.ylabel('Delay Rate')
plt.title('Delay Rate by Origin Airport (Top 15 Busiest)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Breakdown of delay causes
delay_causes = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
cause_totals = df_active[delay_causes].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
cause_totals.plot(kind='bar', color='steelblue')
plt.ylabel('Total Delay Minutes')
plt.title('Total Delay Minutes by Cause')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\nDelay cause breakdown (% of total delay minutes):')
print((cause_totals / cause_totals.sum() * 100).round(1).to_string())

## 7. Correlation Analysis

Correlation measures how strongly two columns move together.
- **+1.0** = perfectly positively correlated
- **-1.0** = perfectly negatively correlated
- **0.0** = no relationship

In [ ]:
num_df = df_active.select_dtypes(include='number')
corr_with_target = num_df.corr()['arr_delay'].drop('arr_delay').sort_values()

plt.figure(figsize=(10, 8))
corr_with_target.plot(kind='barh', color=['#e74c3c' if x > 0 else '#3498db' for x in corr_with_target])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlation with arr_delay')
plt.title('Feature Correlation with Arrival Delay')
plt.tight_layout()
plt.show()

In [ ]:
key_cols = ['dep_delay', 'arr_delay', 'distance', 'air_time',
            'taxi_out', 'taxi_in', 'carrier_delay', 'weather_delay',
            'nas_delay', 'late_aircraft_delay']

plt.figure(figsize=(12, 9))
sns.heatmap(
    df_active[key_cols].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Key Findings & Feature Ideas

In [ ]:
print('=== KEY FINDINGS ===')
print(f'Dataset: {len(df):,} flights in 2024')
print(f'Cancellation rate: {df["cancelled"].mean():.2%}')
print(f'Overall delay rate (>15 min): {delay_rate:.2%}')
print()

print('Most delayed airline:')
print(f'  {carrier_delay.iloc[0]["carrier"]} — {carrier_delay.iloc[0]["delay_rate"]:.1%}')
print()

print('Most delayed hour of day:')
worst_hour = hour_delay.loc[hour_delay['is_delayed'].idxmax()]
print(f'  Hour {int(worst_hour["dep_hour"]):02d}:00 — {worst_hour["is_delayed"]:.1%}')
print()

print('Top delay cause (by total minutes):')
print(f'  {cause_totals.index[0]}: {cause_totals.iloc[0]:,.0f} minutes')
print()

print('Strongest correlates with arr_delay:')
print(corr_with_target.tail(5).to_string())

print()
print('=== FEATURE IDEAS FOR build_features.py ===')
print('  - dep_hour         : extract hour from crs_dep_time')
print('  - is_peak_hour     : 1 if dep_hour in [7,8,17,18,19]')
print('  - is_weekend       : 1 if day_of_week in [6, 7]')
print('  - is_holiday_month : 1 if month in [6,7,8,12]')
print('  - route            : origin + dest combined as single feature')
print('  - dep_hour_bin     : morning / afternoon / evening / night')